# Latent RAG — Factorial Experiment Runner
Compares traditional vs. latent-space RAG across 4 controlled experiments.

| # | Retriever | Generator | Research question |
|---|---|---|---|
| 1 | BGE (dense) | Qwen (text) | Traditional baseline |
| 2 | BGE (dense) | T5Gemma (latent) | Can latent work with good retrieval? |
| 3 | T5Gemma (latent) | Qwen (text) | Does the T5 encoder suck at retrieval? |
| 4 | T5Gemma (latent) | T5Gemma (latent) | Does latent help at all over T5 text? |

## Clone private repo
Use a GitHub Personal Access Token with read access to this repo.

In [8]:
import getpass
import os
import subprocess

os.chdir('/content')

token = getpass.getpass('GitHub token (repo read access): ')
result = subprocess.run(
    ['git', 'clone', f'https://{token}@github.com/zahiTouqan/latent-rag.git'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDOUT:', result.stdout.replace(token, '***'))
    print('STDERR:', result.stderr.replace(token, '***'))
    raise SystemExit('Clone failed')
else:
    print('Clone successful')

Clone successful


In [ ]:
%cd /content/latent-rag
!git fetch
!git checkout v3
!git pull origin v3
!python -m pip install -U pip
!pip install -r requirements.txt
!pip install "datasets>=2.14,<3.0"
!pip install -q --upgrade transformers

## Download some NQ data
Streams `facebook/dpr` NQs + related documents. Adjust the slice `[:100]` to change dataset size.

In [10]:
import json, gzip, urllib.request

url = "https://dl.fbaipublicfiles.com/dpr/data/retriever/biencoder-nq-dev.json.gz"
with urllib.request.urlopen(url) as response:
    with gzip.open(response, 'rt', encoding='utf-8') as f:
        data = json.load(f)

sample = data[:100]

Example DPR record:

In [ ]:
import pprint
first = {k: v for k, v in sample[0].items() if k != 'negative_ctxs'}
first['positive_ctxs'] = [{'title': c['title'], 'passage_id': c['passage_id']} for c in first['positive_ctxs']]
pprint.pprint(first, width=120)

## Build mini corpus from NQ passages
Collects all `positive_ctxs` + `hard_negative_ctxs` from the sampled questions.

In [14]:
import json
from pathlib import Path

CORPUS_OUT = Path("data/passages.jsonl")
CORPUS_OUT.parent.mkdir(parents=True, exist_ok=True)

corpus = {}
for item in sample:
    for p in item["positive_ctxs"] + item["hard_negative_ctxs"]:
        pid = p["passage_id"]
        if pid not in corpus:
            corpus[pid] = {
                "id": pid,
                "text": f"{p['title']}: {p['text']}",
                "doc_id": p["title"]
            }

with CORPUS_OUT.open("w", encoding="utf-8") as f:
    for record in corpus.values():
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(corpus)} passages to {CORPUS_OUT}")

Saved 10010 passages to data/passages.jsonl


## GPU check & HuggingFace login

In [15]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")

CUDA available: True


In [17]:
from huggingface_hub import login

token = getpass.getpass('HF_TOKEN')
login(token)

## Build BGE index (for experiments 1 & 2)
Embedding model: `BAAI/bge-base-en-v1.5` — dense retrieval with [CLS]-token pooling.

In [18]:
BGE_INDEX_DIR = "/content/index_bge"
!python build_index.py --corpus_path {CORPUS_OUT} --index_dir {BGE_INDEX_DIR} --retriever_type bge

Loaded 10010 passages from data/passages.jsonl
Loading BGE embedding model: BAAI/bge-base-en-v1.5
config.json: 100% 777/777 [00:00<00:00, 3.39MB/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 1.72MB/s]
vocab.txt: 232kB [00:00, 9.74MB/s]
tokenizer.json: 711kB [00:00, 13.3MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 669kB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 438M/438M [00:02<00:00, 216MB/s] 
Loading weights: 100% 199/199 [00:00<00:00, 1203.05it/s]
Embedding 10010 passages (40 batches)...
  batch 25/40
  batch 40/40
Saved bge index to /content/index_bge


In [19]:
!ls -lh {BGE_INDEX_DIR}

total 37M
-rw-r--r-- 1 root root  224 May  3 13:55 config.json
-rw-r--r-- 1 root root  30M May  3 13:55 index.faiss
-rw-r--r-- 1 root root 6.8M May  3 13:55 metadata.jsonl


## Build latent (T5) index (for experiments 3 & 4)
Embedding model: `google/t5gemma-2-270m-270m` — mean-pooled encoder latents + safetensors storage.
Stores both FAISS vectors and full sequence latent matrices.
Warning: safetensors file is large (~1.8 GB for 10k passages).

In [21]:
LATENT_INDEX_DIR = "/content/index_latent"
!python build_index.py --corpus_path {CORPUS_OUT} --index_dir {LATENT_INDEX_DIR} --retriever_type latent

Loaded 10010 passages from data/passages.jsonl
Loading latent embedding model (encoder only): google/t5gemma-2-270m-270m
config.json: 6.03kB [00:00, 13.5MB/s]
tokenizer_config.json: 100% 830/830 [00:00<00:00, 3.36MB/s]
tokenizer.json: 100% 33.4M/33.4M [00:01<00:00, 32.6MB/s]
special_tokens_map.json: 100% 662/662 [00:00<00:00, 2.96MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 1.57G/1.57G [00:26<00:00, 59.1MB/s] 
Loading weights: 100% 911/911 [00:00<00:00, 5495.88it/s]
generation_config.json: 100% 195/195 [00:00<00:00, 967kB/s]
Embedding 10010 passages (157 batches)...
  batch 10/157
  batch 20/157
  batch 30/157
  batch 40/157
  batch 50/157
  batch 60/157
  batch 70/157
  batch 80/157
  batch 90/157
  batch 100/157
  batch 110/157
  batch 120/157
  batch 130/157
  batch 140/157
  batch 150/157
  batch 157/157
Saving latent sequences to safetensors...
Saved latent index to /content/index_latent


In [22]:
!ls -lh {LATENT_INDEX_DIR}

total 1.8G
-rw-r--r-- 1 root root  232 May  3 14:08 config.json
-rw-r--r-- 1 root root  25M May  3 14:06 index.faiss
-rw-r--r-- 1 root root 1.8G May  3 14:08 latents.safetensors
-rw-r--r-- 1 root root 6.8M May  3 14:06 metadata.jsonl


## Build eval file
Construct `data/eval.jsonl` from the DPR dev set. `relevant_ids` are document titles for document-level recall.

In [23]:
import json
from pathlib import Path

EVAL_OUT = Path("data/eval.jsonl")

with EVAL_OUT.open("w", encoding="utf-8") as f:
    for item in sample:
        record = {
            "question": item["question"],
            "answer": item["answers"],
            "relevant_ids": [p["title"] for p in item["positive_ctxs"]]
        }
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(sample)} eval examples to {EVAL_OUT}")

Saved 100 eval examples to data/eval.jsonl


## Run factorial experiments
Each cell runs one experiment. Results are saved to `results/` and can be aggregated after all 4 finish.

### Experiment 1: BGE + Text (traditional baseline)
BGE dense retrieval → prompted text generation with Qwen.

In [24]:
!python3 evaluate.py \
    --index_dir {BGE_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode bge+text \
    --top_k 5

Loaded 100 samples from data/eval.jsonl
Loading BGE embedding model: BAAI/bge-base-en-v1.5
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 199/199 [00:01<00:00, 115.57it/s]
Loading text generator model: Qwen/Qwen3.5-0.8B
config.json: 2.91kB [00:00, 828kB/s]
tokenizer_config.json: 16.7kB [00:00, 11.3MB/s]
vocab.json: 6.72MB [00:02, 2.98MB/s]
merges.txt: 3.35MB [00:00, 9.75MB/s]
tokenizer.json: 100% 12.8M/12.8M [00:00<00:00, 18.6MB/s]
chat_template.jinja: 7.75kB [00:00, 16.7MB/s]
model.safetensors.index.json: 50.9kB [00:00, 75.4MB/s]
model.safetensors-00001-of-00001.safeten(…): 100% 1.75G/1.75G [00:20<00:00, 85.6MB/s]
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100% 320/320 [00:00<00:00, 5714.24it/s]
Pipeline mode:

### Experiment 2: BGE + Latent
BGE dense retrieval → T5Gemma re-encodes retrieved passages on the fly → decoder bypass.

In [25]:
!python3 evaluate.py \
    --index_dir {BGE_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode bge+latent \
    --top_k 5

Loaded 100 samples from data/eval.jsonl
Loading BGE embedding model: BAAI/bge-base-en-v1.5
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 199/199 [00:00<00:00, 1318.49it/s]
Loading latent generator model (Seq2Seq): google/t5gemma-2-270m-270m
Loading weights: 100% 911/911 [00:01<00:00, 701.27it/s]
Pipeline mode: bge+latent | Index: 10010 passages from data/passages.jsonl
100% 100/100 [13:14<00:00,  7.94s/it]

=== Results ===
  em                                  0.0000
  f1                                  0.0090
  generated_tokens                    121.9600
  generation_time_s                   7.6073
  latency_p50_ms                      8400.7545
  latency_p95_ms                      8571.1465
  recall@5                            0.6356
  retrieval_time_s                    0.0223
  total_time_s                        7.9394

Saved to results/results_eval_bge+latent_20260503_143730.json


### Experiment 3: T5 + Text
T5 encoder mean-pooled retrieval (likely poor) → Qwen text generation.
Isolates retrieval quality of T5 encoder.

In [26]:
!python3 evaluate.py \
    --index_dir {LATENT_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode t5+text \
    --top_k 5

Loaded 100 samples from data/eval.jsonl
Loading latent embedding model (encoder only): google/t5gemma-2-270m-270m
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 911/911 [00:00<00:00, 4617.53it/s]
Loading text generator model: Qwen/Qwen3.5-0.8B
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100% 320/320 [00:00<00:00, 5102.46it/s]
Pipeline mode: t5+text | Index: 10010 passages from data/passages.jsonl
100% 100/100 [11:06<00:00,  6.66s/it]

=== Results ===
  em                                  0.0000
  f1                                  0.0047
  generated_tokens                    122.0200
  generation_time_s                   6.5759
  latency_p50_ms                      6886.6744
  latency_p95_ms                    

### Experiment 4: T5 + Latent (full latent pipeline)
T5 encoder retrieval + pre-stored safetensors latents → decoder bypass. The current state of the repo.

In [27]:
!python3 evaluate.py \
    --index_dir {LATENT_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode t5+latent \
    --top_k 5

Loaded 100 samples from data/eval.jsonl
Loading latent embedding model (encoder only): google/t5gemma-2-270m-270m
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 911/911 [00:00<00:00, 4638.91it/s]
Loading latent generator model (Seq2Seq): google/t5gemma-2-270m-270m
Loading weights: 100% 911/911 [00:00<00:00, 4822.22it/s]
Pipeline mode: t5+latent | Index: 10010 passages from data/passages.jsonl
100% 100/100 [11:44<00:00,  7.05s/it]

=== Results ===
  em                                  0.0000
  f1                                  0.0025
  generated_tokens                    112.5900
  generation_time_s                   6.9258
  latency_p50_ms                      7977.6978
  latency_p95_ms                      8253.9027
  recall@5                            0.0773
  retrieval_time_s                    0.0704
  total_time_s                        7.0454

Saved to results/results_eval_t5+latent_20260503_150335.json


## Compare results
Collects all result files from `results/` and displays a side-by-side comparison table.

In [28]:
import json
from pathlib import Path

results_dir = Path("results")
files = sorted(results_dir.glob("results_*.json"), key=lambda p: p.stat().st_mtime)

COLUMNS = ["em", "f1", "recall@5", "latency_p50_ms", "latency_p95_ms"]

rows = []
for result_file in files:
    with result_file.open() as f:
        data = json.load(f)
    mode = data["config"]["mode"]
    metrics = data["metrics"]
    rows.append((mode, metrics))

if not rows:
    print("No result files found in results/")
else:
    header = f"{'Experiment':<14s}" + "".join(f"{c:>15s}" for c in COLUMNS)
    print(header)
    print("-" * len(header))
    for mode, m in rows:
        vals = "".join(f"{m.get(c, 0):>15.4f}" for c in COLUMNS)
        print(f"{mode:<14s}{vals}")
    if len(rows) < 4:
        print(f"\n({len(rows)}/4 experiments completed)")

Experiment                 em             f1       recall@5 latency_p50_ms latency_p95_ms
-----------------------------------------------------------------------------------------
bge+text               0.0000         0.0245         0.6356      7236.8325      7514.7023
bge+latent             0.0000         0.0090         0.6356      8400.7545      8571.1465
t5+text                0.0000         0.0047         0.0773      6886.6744      7304.7043
t5+latent              0.0000         0.0025         0.0773      7977.6978      8253.9027


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -lh /content/latent-rag/results
!cp -r /content/latent-rag/results /content/drive/MyDrive/